Standardization: Ensures prompts follow a consistent format (e.g., role, task, constraints).
Guardrails: Embeds safety rules so generated prompts avoid harmful or irrelevant outputs.
Optimization: Encourages token efficiency, modularity, and adaptability.
Automation: Lets you dynamically generate prompts for different tasks without manual rewritin

In [1]:
!pip install --upgrade openai

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 11.6 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.0.6 requires numpy<2,>=1, but you have numpy 2.4.4 which is incompatible.
langchain-openai 0.0.6 requires openai<2.0.0,>=1.10.0, but you have openai 2.38.0 which is incompatible.


In [2]:
import openai
import mysql.connector

# --- MySQL Setup ---
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="prompt_db"
)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS meta_prompts (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(255),
    template TEXT,
    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

# --- Meta-Prompt Template ---
meta_prompt = """
You are a meta-prompt generator.
Your job: create a structured prompt with these sections:
1. Role: Define who the AI should act as.
2. Task: State the exact action required.
3. Constraints: Add safety, formatting, or domain rules.
4. Output: Specify the expected format (JSON, text, table).
Ensure clarity, brevity, and guardrails against unsafe content.
"""

# --- Store Meta-Prompt in MySQL ---
cursor.execute("INSERT INTO meta_prompts (name, template) VALUES (%s, %s)",
               ("Standardized Workflow Prompt", meta_prompt))
conn.commit()

# --- Use Meta-Prompt to Generate a Prompt ---
response = openai.responses.create(
    model="gpt-4.1-mini",
    input=meta_prompt + "\n\nTask: Summarize customer support tickets in JSON format."
)

print(response.output[0].content[0].text)


```plaintext
1. Role: Act as an expert customer support analyst.
2. Task: Summarize the key issue, customer sentiment, and resolution status from each customer support ticket.
3. Constraints: 
   - Do not include any personal or sensitive information.
   - Use clear, concise language suitable for business use.
   - Ensure summaries are neutral and avoid any biased or unsafe content.
4. Output: Provide the summary in JSON format with fields: "ticket_id", "issue_summary", "customer_sentiment" (positive, neutral, negative), and "resolution_status" (resolved, pending, escalated).
```
